In [18]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

In [19]:
# 1. Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [20]:
#load dataset

ds = pd.read_csv('diabetes.csv')
ds.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,tested_positive
1,1,85,66,29,0,26.6,0.351,31,tested_negative
2,8,183,64,0,0,23.3,0.672,32,tested_positive
3,1,89,66,23,94,28.1,0.167,21,tested_negative
4,0,137,40,35,168,43.1,2.288,33,tested_positive


In [21]:
ds.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [22]:
ds.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                      object
dtype: object

In [23]:
ds = pd.get_dummies(ds, columns=['Outcome'], drop_first=True, dtype=int)
ds.rename(columns={'Outcome_tested_positive': 'Outcome'}, inplace=True)


In [24]:
ds.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [25]:
ds.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int32
dtype: object

In [26]:
ds.duplicated().sum()

0

In [27]:
ds.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [29]:
X = ds.drop('Outcome', axis=1)
y = ds['Outcome']

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [30]:
ds.shape

(768, 9)

In [32]:
print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size:  {X_test.shape[0]} samples")
print(f"Positive class ratio (train): {y_train.mean():.2%}")


Training set size: 614 samples
Testing set size:  154 samples
Positive class ratio (train): 34.85%


In [33]:
candidate_models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Machine": SVC(probability=True, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42)
}

scoring_metrics = ['recall', 'precision', 'f1', 'roc_auc', 'average_precision']
benchmark_results = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [34]:
for name, clf in candidate_models.items():
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', clf)
    ])
    
    cv_scores = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=scoring_metrics)
    
    benchmark_results.append({
        'Model': name,
        'Mean Recall': cv_scores['test_recall'].mean(),
        'Mean Precision': cv_scores['test_precision'].mean(),
        'Mean F1': cv_scores['test_f1'].mean(),
        'Mean ROC-AUC': cv_scores['test_roc_auc'].mean(),
        'Mean PR-AUC': cv_scores['test_average_precision'].mean()
    })

results_df = pd.DataFrame(benchmark_results).sort_values(by='Mean Recall', ascending=False)
print("\n=== 5-Fold Stratified CV Benchmark Results ===")
print(results_df.to_string(index=False))


=== 5-Fold Stratified CV Benchmark Results ===
                 Model  Mean Recall  Mean Precision  Mean F1  Mean ROC-AUC  Mean PR-AUC
   Logistic Regression     0.729014        0.636241 0.677941      0.834781     0.742390
Support Vector Machine     0.729014        0.607491 0.660326      0.837283     0.724132
  Gaussian Naive Bayes     0.626357        0.689583 0.655621      0.823728     0.709213
         Random Forest     0.560797        0.694075 0.619263      0.812857     0.696857
   K-Nearest Neighbors     0.542414        0.652145 0.587642      0.782350     0.607301


In [35]:
rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [4, 6, 8, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_
print(f"\nBest Hyperparameters: {grid_search.best_params_}")


Best Hyperparameters: {'classifier__max_depth': 4, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}


In [38]:
y_pred = best_pipeline.predict(X_test)
y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("\n=== Final Test Set Evaluation ===")
print(classification_report(y_test, y_pred, target_names=['Non-Diabetic (0)', 'Diabetic (1)']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Test PR-AUC:  {average_precision_score(y_test, y_pred_proba):.4f}")


MODEL_PATH = "diabetes_model_pipeline.joblib"
joblib.dump(best_pipeline, MODEL_PATH)
print(f"\nPipeline successfully serialized and saved to '{MODEL_PATH}'.")

# Inference Example
loaded_pipeline = joblib.load(MODEL_PATH)
sample_patient = pd.DataFrame([{
    'Pregnancies': 2,
    'Glucose': 148,
    'BloodPressure': 72,
    'SkinThickness': 35,
    'Insulin': 0,      # Will be converted to NaN and imputed automatically
    'BMI': 33.6,
    'DiabetesPedigreeFunction': 0.627,
    'Age': 50
}])

pred = loaded_pipeline.predict(sample_patient)[0]
prob = loaded_pipeline.predict_proba(sample_patient)[0, 1]
print(f"\nSample Patient Prediction: {'Diabetic' if pred == 1 else 'Non-Diabetic'} (Risk Probability: {prob:.2%})")


=== Final Test Set Evaluation ===
                  precision    recall  f1-score   support

Non-Diabetic (0)       0.85      0.77      0.81       100
    Diabetic (1)       0.63      0.74      0.68        54

        accuracy                           0.76       154
       macro avg       0.74      0.76      0.75       154
    weighted avg       0.77      0.76      0.76       154

Confusion Matrix:
[[77 23]
 [14 40]]
Test ROC-AUC: 0.8163
Test PR-AUC:  0.6949

Pipeline successfully serialized and saved to 'diabetes_model_pipeline.joblib'.

Sample Patient Prediction: Diabetic (Risk Probability: 74.28%)
